# 4장 1강 · 이미지 임베딩과 특징 표현

실습파일  |  Oxford-IIIT Pet · ResNet18

비글 사진 한 장을 입력하면 어떤 사진을 첫 번째로 보여 주어야 할까요? 이 실습에서는 같은 사진을 픽셀과 임베딩으로 각각 표현하고, 검색 기준을 바꾸었을 때 실제로 어떤 사진이 1위에 오는지 확인합니다.

## 실습 안내

| 구간 | 풀어 볼 질문 | 산출물 |
|---|---|---|
| 필수 1 | 해상도가 다른 사진이 어떻게 길이 512인 벡터가 될까? | 전처리 검산, 층별 출력, ID와 특징값 표 |
| 필수 2 | 코사인 유사도와 유클리드 거리는 언제 같은 순위를 만들까? | 손계산, 실제 순위 역전, 단위 벡터 검증 |
| 과제 1 | 이 표현은 품종 검색과 분류에 얼마나 도움이 될까? | 검색 성적표, 학습한 분류기, 오답 사례 해석 |

**준비물**  강의에 제공된 Oxford ZIP을 압축 상태로 노트북과 같은 폴더에 둡니다. 루트에 `metadata.csv`가 있고 `image_path` 열이 ZIP 내부 사진 경로를 가리키는 배포본을 사용합니다. 파일 이름이 다르면 아래 `paths`를 수정하세요. Python과 Jupyter, NumPy, pandas, Pillow, 서로 호환되는 PyTorch와 torchvision이 필요합니다. 검증 환경은 CPU용 PyTorch 2.6과 torchvision 0.21입니다. ResNet18 기본 가중치는 첫 실행 때 내려받으며, 이후에는 PyTorch 캐시를 사용합니다.

**실행 순서**  공통 준비 ①~③을 먼저 실행합니다. 필수 1, 필수 2, 과제 1은 각각 공통 준비 직후부터 시작할 수 있습니다. 각 구간 안에서는 번호순으로 진행하세요. 표에서 ID 하나는 사진 한 장을 뜻합니다. 점수를 반올림하는 작업은 화면에 표시할 때만 합니다.

### 공통 준비 ① · 어떤 사진을 어디에 쓸까?

8개 품종에서 검색 후보(갤러리) 8장씩, 검색을 요청할 사진(질의) 6장씩을 뽑습니다. 분류기 학습용으로는 품종당 12장을 따로 뽑습니다. 고정 예제 7장을 더하면 총 215장입니다. 배포본의 `val` 중 96장은 이 수업의 분류기 학습용, `test` 중 112장은 갤러리와 질의용입니다. 이 실험에서는 별도 하이퍼파라미터 탐색을 하지 않습니다.

`metadata.csv`의 모든 행에 원본 사진이 들어 있는 것은 아니므로, ZIP 안에 실물이 있는 행만 사용합니다. 정렬한 뒤 고정 난수로 비복원 추출하고 파일 ID와 RGB 내용 중복을 검사합니다. `photos[id]`는 원본 사진, `pos[id]`는 아래 특징 행렬에서 그 사진이 놓일 행 번호입니다.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import hashlib
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display

torch.set_num_threads(2)
SEED = 42
paths = [Path("oxford_dataset.zip"), Path("oxford_dataset.zip")]
data_path = next((p for p in paths if p.is_file()),None)
if data_path is None:
    raise FileNotFoundError("Oxford ZIP을 노트북 옆에 두거나 paths에 실제 경로를 넣으세요.")
breeds = ["beagle","pug","boxer","shiba_inu","Persian","Birman","Ragdoll","Siamese"]
example_ids = ["beagle_110","beagle_111","Persian_100"]
pug_ids = ["pug_100","pug_103","beagle_117","Persian_125"]
demo_ids = example_ids + pug_ids
with ZipFile(data_path) as z:
    meta = pd.read_csv(z.open("metadata.csv")).set_index("image_id")
    meta = meta.loc[meta.image_path.isin(z.namelist()) & meta.class_name.isin(breeds)].sort_index()
rng = np.random.default_rng(SEED)
train_ids, gallery_ids, query_ids = [], [], []
for name in breeds:
    subset = meta[meta.class_name == name]
    pool = subset[(subset.split == "val") & ~subset.index.isin(demo_ids)].index
    train_ids.extend(rng.choice(pool,12,replace=False).tolist())
    chosen = rng.choice(subset[subset.split=="test"].index,14,replace=False).tolist()
    gallery_ids.extend(chosen[:8])
    query_ids.extend(chosen[8:])
all_ids = demo_ids + train_ids + gallery_ids + query_ids
assert len(all_ids) == len(set(all_ids))
photos, hashes = {}, []
with ZipFile(data_path) as z:
    for key in all_ids:
        with Image.open(BytesIO(z.read(meta.loc[key,"image_path"]))) as source:
            im = source.convert("RGB")
        photos[key] = im
        digest = hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
        hashes.append(digest)
assert len(set(hashes))==len(all_ids), "크기와 RGB 값이 동일한 사진이 있습니다."
pos = {key:i for i,key in enumerate(all_ids)}

def show_photos(ids):
    sheet = Image.new("RGB",(185*len(ids),180),"white")
    pen = ImageDraw.Draw(sheet)
    for i,key in enumerate(ids):
        im = ImageOps.contain(photos[key],(180,145))
        sheet.paste(im,(185*i+(180-im.width)//2,0))
        pen.text((185*i+3,150),key,fill="black")
    display(sheet)
print("예제 / 분류기 학습 / 검색 후보 / 질의",*[len(g) for g in [demo_ids,train_ids,gallery_ids,query_ids]])

예제 / 분류기 학습 / 검색 후보 / 질의 7 96 64 48


### 공통 준비 ② · 같은 사진에서 두 표현 만들기

`weights.transforms()`는 해당 가중치의 전처리 순서를 가져옵니다. 사진들을 같은 크기로 준비하고 16장 이하로 모아 모델에 넣습니다. `encoder.eval()`은 모델의 배치 정규화 층이 학습 때 저장한 통계를 쓰게 합니다. `inference_mode()`는 특징 추출 중 기울기 기록을 생략합니다. 원래 분류층은 비교를 위해 `original_fc`에 보관합니다.

`E`의 한 행은 512개 모델 특징입니다. `P`의 한 행은 같은 전처리 영상을 16×16으로 평균 축소해 펼친 768개 RGB 값입니다. `pick(E, ids)`는 주어진 ID 순서대로 행을 꺼내고, `unit`은 각 행을 그 행의 길이로 나눕니다. 뒤 문항에서 이 변환을 수치로 검산합니다.

In [2]:
weights = ResNet18_Weights.DEFAULT
preprocess = weights.transforms()
encoder = resnet18(weights=weights).eval()
original_fc = encoder.fc
encoder.fc = nn.Identity()
encoder.requires_grad_(False)

def extract(ids):
    embedded, pixels = [], []
    with torch.inference_mode():
        for start in range(0,len(ids),16):
            batch = torch.stack([preprocess(photos[key]) for key in ids[start:start+16]])
            embedded.append(encoder(batch).cpu().numpy())
            # 같은 크롭·같은 정규화에서 RGB만 16×16으로 평균 축소한 비교 기준
            pooled = nn.functional.adaptive_avg_pool2d(batch,(16,16))
            pixels.append(pooled.flatten(1).numpy())
    return np.concatenate(embedded), np.concatenate(pixels)

E, P = extract(all_ids)
def pick(matrix,ids):
    return matrix[[pos[k] for k in ids]]
def unit(x):
    lengths = np.linalg.norm(x,axis=1,keepdims=True)
    if np.any(lengths==0):
        raise ValueError("길이가 0인 행은 코사인 비교를 할 수 없습니다.")
    return x/lengths
print("RGB 픽셀 특징",P.shape,"ResNet 임베딩",E.shape)

RGB 픽셀 특징 (215, 768) ResNet 임베딩 (215, 512)


### 공통 준비 ③ · 검색과 채점 도구

`neighbors(특징행렬, 질의ID목록, 후보ID목록, 기준)`은 점수표와 순위 인덱스를 반환합니다. 점수표의 행은 질의, 열은 후보입니다. 순위 인덱스의 첫 열은 각 질의에서 이긴 **후보의 열 번호**입니다. 코사인은 큰 값부터, 거리는 작은 값부터 정렬합니다. 같은 값이면 원래 후보 순서를 유지합니다. `eval_ranks`는 완성된 순위에 품종 라벨을 붙여 채점합니다. 같은 품종을 관련 사진으로 정의하는 수업용 기준입니다.

In [3]:
def neighbors(matrix, queries, gallery, metric="cosine"):
    q = pick(matrix,queries).astype(np.float64)
    g = pick(matrix,gallery).astype(np.float64)
    if metric=="cosine":
        values = unit(q) @ unit(g).T
        order = np.argsort(-values,axis=1,kind="stable")
    elif metric=="euclidean":
        values = np.stack([np.linalg.norm(g-v,axis=1) for v in q])
        order = np.argsort(values,axis=1,kind="stable")
    else:
        raise ValueError(metric)
    return values, order

def eval_ranks(order,queries,gallery):
    # 품종 라벨은 순위 결정이 끝난 뒤에 사용합니다.
    relevance = meta.loc[gallery,"class_name"].to_numpy()[order] == meta.loc[queries,"class_name"].to_numpy()[:,None]
    return pd.DataFrame({"query":queries, "breed":meta.loc[queries,"class_name"].to_numpy(),
                         "top1":np.array(gallery)[order[:,0]], "hit1":relevance[:,0],
                         "hit3":relevance[:,:3].any(axis=1),
                         "correct_in3":relevance[:,:3].sum(axis=1)})

## 필수 1 · 원본 사진에서 512차원 임베딩까지

비글 두 장과 페르시안 한 장을 따라갑니다. 행은 어떤 사진인지, 열은 어떤 종류의 수치인지 구분해 설명하세요.

### 1-1. 원본과 모델 입력의 픽셀 대조

예제 3장의 원본, 품종과 가로·세로 크기를 확인하세요. 권장 전처리로 배치를 만들고 첫 사진의 중심 크롭에서 왼쪽 위 픽셀의 빨간 채널을 직접 표준화해 모델 입력과 비교하세요. 원본과 크롭에서 같은 좌표가 같은 지점을 뜻하는지도 설명하세요.

**풀이 힌트**  
`preprocess.resize_size`와 `crop_size`를 확인하고 torchvision의 resize, center_crop을 순서대로 사용합니다. 8비트 값을 255로 나눈 뒤 빨간 채널의 평균과 표준편차를 적용하세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# 1-1 · 원본과 모델 입력의 픽셀 대조

**나의 해석**  
관찰한 값과 그 값으로 설명할 수 있는 내용을 적으세요.

✎

### 1-2. 특징 맵에서 512개 숫자가 만들어지는 지점

같은 배치를 ResNet18의 마지막 합성곱 블록, 평균 풀링, 평탄화, 분류층 순서로 추적하세요. 첫 사진·첫 채널의 7×7 평균을 직접 구하고, 저장해 둔 특징과 원래 분류층을 적용한 출력의 모양도 확인하세요.

**풀이 힌트**  
`list(encoder.children())[:-2]`는 마지막 풀링과 fc 앞까지의 층을 가져옵니다. `flatten(1)`은 사진 축을 남깁니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# 1-2 · 특징 맵에서 512개 숫자가 만들어지는 지점

**나의 해석**  
관찰한 값과 그 값으로 설명할 수 있는 내용을 적으세요.

✎

### 1-3. 행 순서가 바뀌어도 사진의 특징을 찾을 수 있을까?

사진을 역순으로 다시 인코딩하고 ID별 첫 다섯 특징을 표로 표시하세요. 원래 순서의 행과 바로 비교한 결과, ID로 맞춘 결과를 각각 검산하세요.

**풀이 힌트**  
`example_ids[::-1]`는 역순 목록입니다. `pick`을 호출할 때 그 목록을 넘기면 기존 행렬에서도 같은 순서로 꺼낼 수 있습니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# 1-3 · 행 순서가 바뀌어도 사진의 특징을 찾을 수 있을까?

**나의 해석**  
관찰한 값과 그 값으로 설명할 수 있는 내용을 적으세요.

✎

### 1-4. 특징 한 개가 비슷하면 같은 품종일까?

beagle_110과 나머지 두 사진을 비교하세요. 0번 특징의 절댓값 차이와 512개 전체의 코사인 유사도를 나란히 구하고, 각각 어느 사진을 고르는지 설명하세요.

**풀이 힌트**  
차이는 0에 가까울수록, 코사인은 클수록 가깝습니다. `neighbors`의 첫 반환값은 입력 후보 순서에 대응합니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# 1-4 · 특징 한 개가 비슷하면 같은 품종일까?

**나의 해석**  
관찰한 값과 그 값으로 설명할 수 있는 내용을 적으세요.

✎

## 필수 2 · 거리와 유사도가 바꾸는 검색 순위

코사인 유사도는 두 벡터 사이 각도의 코사인입니다. 유클리드 거리는 두 끝점 사이의 직선거리입니다. 작은 벡터로 공식을 검산하고 실제 사진에서 두 기준이 다른 이웃을 고르는 경우를 찾아보세요.

### 2-1. 두 성분으로 손계산하고 길이를 바꿔 보기

교안의 운동화 A=(1,0), 운동화 B=(0.8,0.6), 가방 C=(0,1)를 사용하세요. A에서 B와 C까지의 내적·길이·코사인·거리를 계산하고, B를 세 배로 늘리면 어느 지표의 순위가 바뀔지 예측한 뒤 확인하세요.

**풀이 힌트**  
내적은 대응 성분의 곱을 더한 값입니다. 두 벡터 길이의 곱으로 나누면 코사인이 됩니다. 거리는 성분 차이 제곱합의 제곱근입니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# 2-1 · 두 성분으로 손계산하고 길이를 바꿔 보기

**나의 해석**  
관찰한 값과 그 값으로 설명할 수 있는 내용을 적으세요.

✎

### 2-2. 비글 사진의 유사도를 중간값부터 계산하기

beagle_110과 나머지 예제 두 장의 내적, 양쪽 벡터 길이, 코사인, 유클리드 거리를 구하세요. 직접 계산한 값이 검색 함수의 값과 일치하는지 검산하고 1위 사진을 고르세요.

**풀이 힌트**  
벡터 길이는 제곱합의 제곱근입니다. 파일 ID로 1차원 벡터를 꺼내 사진 한 쌍씩 계산하면 중간값을 보기 쉽습니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# 2-2 · 비글 사진의 유사도를 중간값부터 계산하기

**나의 해석**  
관찰한 값과 그 값으로 설명할 수 있는 내용을 적으세요.

✎

### 2-3. 실제 후보 64장 속에서 1위가 바뀌는 이유

beagle_86을 질의로 사용해 코사인과 거리의 상위 3장을 각각 출력하세요. 후보 ID, 후보 벡터 길이, 두 점수를 같은 표에서 비교하세요. 질의 48장 중 두 방법의 1위 파일이 다른 질의도 세어 보세요.

**풀이 힌트**  
두 기준을 같은 후보 열에 적용합니다. `order[:,0]`끼리 비교하면 모든 질의의 1위 변경 여부를 한꺼번에 계산할 수 있습니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# 2-3 · 실제 후보 64장 속에서 1위가 바뀌는 이유

**나의 해석**  
관찰한 값과 그 값으로 설명할 수 있는 내용을 적으세요.

✎

### 2-4. 길이를 1로 맞춘 뒤 두 순위 검증하기

모든 임베딩을 길이 1로 맞추고 48×64개 쌍에서 단위 벡터 거리의 제곱과 2×(1−코사인)을 대조하세요. 두 기준이 만든 전체 순위의 일치 여부와 오차를 출력하고 이유를 설명하세요.

**풀이 힌트**  
정규화 전후 비교에서 같은 질의·후보 순서를 사용하세요. 정규화하기 전에 float64로 변환하고 두 행렬의 절댓값 차이의 최댓값을 출력합니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# 2-4 · 길이를 1로 맞춘 뒤 두 순위 검증하기

**나의 해석**  
관찰한 값과 그 값으로 설명할 수 있는 내용을 적으세요.

✎

## 과제 1 · 품종 검색 성적표와 분류기 만들기

반려동물 사진 서비스에 유사 사진 3장을 보여 주려고 합니다. 이제 공통 준비의 특징을 사용해 새 질의를 검색하고, 같은 특징을 사용하는 품종 분류기도 학습하세요. 과제는 3-1부터 차례로 풉니다.

### 3-1. 새 퍼그 사진의 이웃을 직접 판정하기

pug_100을 질의로, pug_103·beagle_117·Persian_125를 후보로 사용하세요. 사진을 나란히 보고 코사인·거리의 1위를 각각 표시하세요. 두 지표의 근거 수치와 사진에서 확인할 수 있는 단서를 구분해 적으세요.

**풀이 힌트**  
첫 ID만 질의 목록에 넣고 나머지 세 ID를 후보 목록에 넣습니다. 품종 이름을 보고 순위를 정하지 않습니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# 3-1 · 새 퍼그 사진의 이웃을 직접 판정하기

**나의 해석**  
관찰한 값과 그 값으로 설명할 수 있는 내용을 적으세요.

✎

### 3-2. 같은 질의로 세 검색 방법 평가하기

48장×64장으로 RGB 픽셀+코사인, 임베딩+코사인, 임베딩+원래 거리의 성능을 비교하세요. 1위 적중률, 3위 이내 적중률, Precision@3를 분자·분모와 함께 계산하세요. 3장 중 하나만 맞히면 되는 서비스와 관련 사진을 많이 보여 줄 서비스에서 어떤 결과를 택할지 쓰세요.

**풀이 힌트**  
`eval_ranks`의 hit1과 hit3은 질의별 참·거짓입니다. correct_in3은 각 질의의 1~3위 안에 들어간 같은 품종 사진 개수(0~3)입니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# 3-2 · 같은 질의로 세 검색 방법 평가하기

**나의 해석**  
관찰한 값과 그 값으로 설명할 수 있는 내용을 적으세요.

✎

### 3-3. 같은 임베딩으로 8개 품종 분류기 학습하기

학습 사진 96장의 임베딩을 단위 벡터로 바꾸고 품종명을 0~7의 정수로 변환하세요. Linear(512,8) 층만 Adam(학습률 0.02)으로 40회 갱신하세요. 첫·마지막 갱신 직전 손실, 가중치 변화와 학습에 쓰지 않은 질의 48장의 정확도를 제시하세요.

**풀이 힌트**  
모델 입력은 float32, 정답은 정수 인덱스입니다. 반복 순서는 zero_grad → 순전파 → cross_entropy → backward → step입니다. 질의에서는 기울기 계산을 끄고 점수가 가장 큰 열 번호를 품종명으로 되돌리세요.

In [ ]:
# 여기에 풀이를 작성하세요.
# 3-3 · 같은 임베딩으로 8개 품종 분류기 학습하기

**나의 해석**  
관찰한 값과 그 값으로 설명할 수 있는 내용을 적으세요.

✎

### 3-4. 1위가 틀린 사진을 추적하고 결과 보고하기

boxer_57의 코사인 상위 세 사진을 점수순으로 보여 주세요. 이 질의의 Hit@1, Hit@3, Precision@3와 분류기의 예측을 판정하세요. 마지막으로 48장 전체에서 검색 1위와 분류기의 정오를 교차 집계하고, 결과를 해석할 때의 한계를 한 가지 쓰세요.

**풀이 힌트**  
검색과 분류 결과를 같은 query_ids 순서로 맞추세요. 행을 분류 정오, 열을 검색 정오로 둔 교차표는 pd.crosstab으로 만들 수 있습니다.

In [ ]:
# 여기에 풀이를 작성하세요.
# 3-4 · 1위가 틀린 사진을 추적하고 결과 보고하기

**나의 해석**  
관찰한 값과 그 값으로 설명할 수 있는 내용을 적으세요.

✎

## 실습 마무리

1. 비글 한 장이 길이 512인 벡터가 되기까지 크기와 축이 어떻게 변했는지 설명하세요.
2. “코사인 0.83”을 읽을 때 어떤 결론을 내릴 수 있는지, 1위 후보가 거리 기준에서 바뀔 수 있는 이유와 함께 쓰세요.
3. 관련 사진 한 장을 찾는 서비스와 상위 세 자리를 채우는 서비스 중 하나를 골라, 실제 분자·분모로 검색 기준을 제안하세요.
4. boxer_57을 예로 들어 사진 검색과 품종 분류가 서로 다른 답을 낼 수 있는 이유를 쓰세요.

**제출할 내용**  각 문항의 코드와 실행 결과, 위 네 질문에 대한 해석을 자신의 말로 작성하세요.

1. …
2. …
3. …
4. …